# Tool Execution Strategy: Parallel vs Sequential

When Claude calls multiple tools, execution strategy comes down to one question:
**does Tool B need Tool A's output as its input?**

| Situation | Strategy | Round trips |
|---|---|---|
| Tools are **independent** (no data dependency) | **Parallel** | Fewer |
| Tool B **depends on** Tool A's result | **Sequential** | More — but necessary |

Claude 4 natively returns multiple `tool_use` blocks in a single response when tools are independent.
This notebook shows both patterns with the **same agentic loop** — Claude's behavior changes based on
data dependencies, not based on how the loop is coded.

In [ ]:
import Anthropic from 'npm:@anthropic-ai/sdk';
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const client = new Anthropic({ apiKey: Deno.env.get('ANTHROPIC_API_KEY') ?? '' });
const MODEL = 'claude-sonnet-4-6';

console.log('Setup complete. Model:', MODEL);

## The Data Dependency Rule

Before choosing parallel or sequential, ask:

> **Can Tool B start before Tool A finishes?**

- **No dependency** → **Parallel** — Claude 4 returns both `tool_use` blocks in one turn
- **Dependency exists** → **Sequential** — Claude returns one `tool_use` per turn, by necessity

| Scenario | Dependency? | Strategy |
|---|---|---|
| Fetch domain description + fetch domain weight (same query) | None | Parallel |
| Identify weakest domain → fetch study resources for it | Yes (B needs A's output) | Sequential |
| Search web + read local file | None | Parallel |
| Search → retrieve top result → summarize retrieved text | Chain | Sequential |

In [ ]:
// Independent tools — each answers a different aspect of the same query; neither needs the other's result
function getDomainInfo(domain: string): string {
  const info: Record<string, string> = {
    D1: 'Agentic Architecture & Orchestration — orchestrator/subagent design, agentic loops, hooks.',
    D2: 'Tool Design & MCP Integration — tool schema design, MCP primitives, transport selection.',
    D3: 'Claude Code Configuration & Workflows — CLAUDE.md hierarchy, custom commands.',
    D4: 'Prompt Engineering & Structured Output — JSON mode, null handling, format normalization.',
    D5: 'Context Management & Reliability — prompt caching, batch API, context compaction.',
  };
  return info[domain] ?? 'No info found for domain: ' + domain;
}

function getDomainWeight(domain: string): string {
  const weights: Record<string, string> = {
    D1: '~25% of the exam — highest-weight domain.',
    D2: '~20% of the exam.',
    D3: '~20% of the exam.',
    D4: '~20% of the exam.',
    D5: '~15% of the exam.',
  };
  return weights[domain] ?? 'No weight data for domain: ' + domain;
}

// Dependent tools — get_study_resources cannot run until identify_weak_domain returns a result
function identifyWeakDomain(scoresJson: string): string {
  const scores = JSON.parse(scoresJson) as Record<string, number>;
  const sorted = Object.entries(scores).sort((a, b) => a[1] - b[1]);
  return sorted[0][0];
}

function getStudyResources(domain: string): string {
  const resources: Record<string, string> = {
    D1: 'claude_agent_sdk/00-03 notebooks; managed_agents/CMA_* series.',
    D2: 'tool_use/* notebooks; mcp/01_mcp_primitives; mcp/03_tool_schema_design.',
    D3: 'claude_agent_sdk/01_claude_code_configuration + SkillJar courses.',
    D4: 'null_handling_and_normalization; schema_validation_retry; semantic_error_detection.',
    D5: 'batch_processing; context_management_guide; application_layer_hooks notebooks.',
  };
  return resources[domain] ?? 'No resources for domain: ' + domain;
}

function dispatch(name: string, input: Record<string, string>): string {
  switch (name) {
    case 'get_domain_info':      return getDomainInfo(input.domain);
    case 'get_domain_weight':    return getDomainWeight(input.domain);
    case 'identify_weak_domain': return identifyWeakDomain(input.scores_json);
    case 'get_study_resources':  return getStudyResources(input.domain);
    default: throw new Error('Unknown tool: ' + name);
  }
}

const domainInfoTool: Anthropic.Tool = {
  name: 'get_domain_info',
  description: 'Returns the description and scope of a CCA-F knowledge domain.',
  input_schema: {
    type: 'object',
    properties: {
      domain: { type: 'string', description: 'Domain code: D1, D2, D3, D4, or D5.' },
    },
    required: ['domain'],
  },
};

const domainWeightTool: Anthropic.Tool = {
  name: 'get_domain_weight',
  description: 'Returns the exam coverage percentage for a CCA-F knowledge domain.',
  input_schema: {
    type: 'object',
    properties: {
      domain: { type: 'string', description: 'Domain code: D1, D2, D3, D4, or D5.' },
    },
    required: ['domain'],
  },
};

const identifyWeakDomainTool: Anthropic.Tool = {
  name: 'identify_weak_domain',
  description: 'Given a JSON-stringified object mapping domain codes to practice-exam scores (0-100), returns the domain code with the lowest score.',
  input_schema: {
    type: 'object',
    properties: {
      scores_json: {
        type: 'string',
        description: 'JSON-stringified object mapping domain codes to integer scores, e.g. the result of JSON.stringify({D1:70,D2:45,D3:80,D4:90,D5:60}).',
      },
    },
    required: ['scores_json'],
  },
};

const getStudyResourcesTool: Anthropic.Tool = {
  name: 'get_study_resources',
  description: 'Returns recommended study notebooks and resources for a specific CCA-F domain.',
  input_schema: {
    type: 'object',
    properties: {
      domain: { type: 'string', description: 'Domain code: D1, D2, D3, D4, or D5.' },
    },
    required: ['domain'],
  },
};

console.log('Tools defined.');

## The Agentic Loop

One loop handles both patterns. The two rules:

1. Execute all `tool_use` blocks from a single turn with `Promise.all()` (concurrent)
2. Return **all** `tool_result` blocks together in **a single user message**

For **independent tools**, Claude 4 returns multiple `tool_use` blocks in one turn — `Promise.all()` runs them concurrently.

For **dependent tools**, Claude can only emit one `tool_use` per turn (it has no output from the first tool yet),
so the loop naturally sequences them across multiple round trips.

The loop code is identical for both cases; Claude's judgment about data dependencies drives the difference.

```
// Correct — all results in one message (preserves parallel execution)
messages.push({ role: 'user', content: [result1, result2] });

// Wrong — separate messages disable parallel execution
messages.push({ role: 'user', content: [result1] });
messages.push({ role: 'user', content: [result2] });
```

### How `.map()` and `Promise.all()` split the work

The two operators below do **two different jobs**. Reading them as one line hides that — so the loop splits them into `pendingResults` (fire) and `results` (collect):

- **`.map()` = FIRE.** It walks `toolUseBlocks` synchronously and calls each `async` callback. Calling an `async` function *starts* its work and immediately hands back a `Promise` — `.map()` never waits for one to finish before starting the next. So all tools are launched effectively *together*. `.map()` itself isn't "parallel" magic; the parallelism comes from `async` returning without waiting.
- **`Promise.all()` = COLLECT.** Each tool finishes at a **different speed** (a 2s web call vs a 50ms file read), so right after `.map()` we hold an array of *unsettled* `Promise`s, not values. `Promise.all()` waits for **every** one to settle, then returns the results as an array in the **original order**.

```
toolUseBlocks
     |
     |-- tc[0] get_domain_info    --async fire-->  Promise(pending) --|
     |-- tc[1] get_domain_weight  --async fire-->  Promise(pending) --|
     |                                                                |
     |                       (.map finished — all tools launched)     |
     |                                                                |
     +----------------------  await Promise.all  ---------------------+
                                       |
                                       v
                            [ result0, result1 ]   (ordered, all settled)
```

> **One-liner:** `.map()` *launches* every tool at once; `Promise.all()` *waits* for the slowest and gathers the results in order.


In [ ]:
async function agentLoop(
  userQuery: string,
  tools: Anthropic.Tool[],
  label: string,
  disableParallel = false
): Promise<void> {
  const messages: Anthropic.MessageParam[] = [{ role: 'user', content: userQuery }];
  let roundTrips = 0;

  console.log('');
  console.log('='.repeat(60));
  console.log(label);
  console.log('Query: ' + userQuery);
  console.log('');

  while (true) {
    const response = await client.messages.create({
      model: MODEL,
      max_tokens: 1024,
      tools,
      tool_choice: { type: 'auto', disable_parallel_tool_use: disableParallel },
      messages,
    });

    roundTrips++;

    const toolUseBlocks = response.content.filter(
      (b): b is Anthropic.ToolUseBlock => b.type === 'tool_use'
    );

    console.log(
      'Round trip ' + roundTrips +
      ' | stop_reason: ' + response.stop_reason +
      ' | tool_use blocks this turn: ' + toolUseBlocks.length
    );

    for (const block of response.content) {
      if (block.type === 'text') {
        console.log('  Claude: ' + block.text.slice(0, 200));
      }
      if (block.type === 'tool_use') {
        console.log('  Tool call: ' + block.name + '(' + JSON.stringify(block.input) + ')');
      }
    }

    if (response.stop_reason === 'end_turn') break;

    messages.push({ role: 'assistant', content: response.content });

    if (toolUseBlocks.length > 0) {
      // ── Step 1: .map() FIRES each tool call — it does NOT wait ───────────────
      // Calling an `async` callback starts its work immediately and returns a
      // Promise right away. .map() never pauses for one before starting the next,
      // so every tool is launched back-to-back — effectively "all at once".
      //
      //   toolUseBlocks
      //        |
      //        |-- tc[0] get_domain_info    --async fire-->  Promise(pending) --|
      //        |-- tc[1] get_domain_weight  --async fire-->  Promise(pending) --|
      //        |                                                                |
      //   pendingResults = [ Promise(pending), Promise(pending) ]   (NOT values yet)
      //
      const pendingResults: Promise<Anthropic.ToolResultBlockParam>[] =
        toolUseBlocks.map(async (tc) => ({
          type: 'tool_result' as const,
          tool_use_id: tc.id,
          content: dispatch(tc.name, tc.input as Record<string, string>),
        }));

      // ── Step 2: Promise.all() COLLECTS them ─────────────────────────────────
      // The tools run concurrently but finish at DIFFERENT speeds (e.g. a 2s web
      // call vs a 50ms file read), so right now we only hold Promises, not values.
      // Promise.all waits until EVERY Promise settles, then returns the results as
      // an array — in the original order, regardless of who finished first.
      //
      //   Promise(pending) --|
      //   Promise(pending) --|--  await Promise.all  -->  [ result0, result1 ]
      //
      const results: Anthropic.ToolResultBlockParam[] = await Promise.all(pendingResults);

      // Return ALL results together in ONE user message (preserves parallel execution)
      messages.push({ role: 'user', content: results });
    }
  }

  console.log('');
  console.log('Total round trips: ' + roundTrips);
}

console.log('agentLoop defined.');

## Section 1: Parallel Execution — Independent Tools

The query asks for **two different aspects of the same domain** — description and weight.
Neither answer depends on the other; both can be fetched simultaneously.

Expected: Claude 4 returns **both `tool_use` blocks in a single turn** (round trip 1),
resolving the query in **2 round trips** total.

In [ ]:
await agentLoop(
  'Tell me about domain D2 — what is it, and how much of the exam does it cover?',
  [domainInfoTool, domainWeightTool],
  'PARALLEL — Independent tools (no data dependency)'
);

## Section 2: Sequential Execution — Dependent Tools

The query asks to **first identify the weakest domain, then recommend resources for it**.
`get_study_resources` cannot run until `identify_weak_domain` returns which domain to look up.

Expected: Claude runs one tool per turn across **3 round trips** total
(identify → resources → final answer) — the extra round trips are necessary, not wasteful.

In [ ]:
await agentLoop(
  'My CCA-F practice scores: D1=70, D2=45, D3=80, D4=90, D5=60. Find my weakest domain and recommend how to study for it.',
  [identifyWeakDomainTool, getStudyResourcesTool],
  'SEQUENTIAL — Dependent tools (get_study_resources needs identify_weak_domain output)'
);

## Section 3: Controlled Parallelism — `disable_parallel_tool_use`

Sections 1 and 2 showed patterns driven by **data dependency** — Claude decides whether to batch
or sequence based on whether Tool B needs Tool A's output.

`disable_parallel_tool_use` lets you **override that decision** and force at most one `tool_use`
block per turn, regardless of whether tools are independent.

Pass it as a field inside `tool_choice`:

```typescript
tool_choice: { type: 'auto', disable_parallel_tool_use: true }
```

### Effect by `tool_choice` type

| `tool_choice.type` | `disable_parallel_tool_use` | Claude emits per turn |
|---|---|---|
| `auto` | `false` (default) | 0, 1, or many `tool_use` blocks |
| `auto` | `true` | **at most one** `tool_use` block |
| `any` or `tool` | `true` | **exactly one** `tool_use` block |

> This flag only constrains Claude's **output** — how many `tool_use` blocks it emits.
> It does not control how you execute tools on your side.

### When to use

- Tools have side effects that must not run concurrently (e.g. writes that could conflict)
- Strict execution ordering is required for auditing or debugging
- You want to simplify your handler to always process exactly one result at a time

**Expected:** The same independent-tools query from Section 1 now takes **3 round trips** instead
of 2 — the deliberate cost of forcing sequential behavior on tools with no actual data dependency.

In [ ]:
// Same independent-tools query as Section 1 — but with disable_parallel_tool_use: true.
// Claude would normally batch both tool_use blocks in one turn (2 round trips).
// The flag forces at most one tool_use per turn: expected 3 round trips instead of 2.
await agentLoop(
  'Tell me about domain D2 — what is it, and how much of the exam does it cover?',
  [domainInfoTool, domainWeightTool],
  'FORCED SEQUENTIAL — disable_parallel_tool_use: true (same query as Section 1)',
  true,
);

## Summary

### Decision framework

```
Can Tool B start before Tool A finishes?
|
+-- NO  (independent) --> Parallel (default)
|       Claude 4 returns multiple tool_use blocks in one turn
|       Execute with Promise.all()
|       Return ALL tool_result blocks in ONE user message
|
+-- YES (dependent)  --> Sequential
|       Claude returns one tool_use per turn — it cannot batch what it doesn't know yet
|       Loop runs naturally across multiple round trips
|       Same handler code; Claude controls the pace
|
+-- Need guaranteed ordering? --> Forced Sequential
        { type: 'auto', disable_parallel_tool_use: true } in tool_choice
        Claude emits at most one tool_use per turn regardless of dependencies
        Use when side effects must not overlap, or strict ordering is required
```

### Comparison

| | Parallel | Sequential | Forced Sequential |
|---|---|---|---|
| Tool dependency | None | B needs A's output | None (overridden) |
| What drives it | Claude's inference | Data dependency | `disable_parallel_tool_use: true` |
| Claude's behavior | Multiple `tool_use` blocks in one turn | One `tool_use` per turn | One `tool_use` per turn |
| Your handler | `Promise.all()` — all results in one message | Same loop — one at a time | Same loop — one at a time |
| Round trips | Fewer | More (necessary) | More (deliberate cost) |
| Example | description + weight for same entity | identify domain → fetch resources | Side-effectful writes, audit logging |

### Related notebooks

- `tool_use/tool_choice.ipynb` — `auto` / `any` / `tool` mode differences
- `tool_use/calculator_tool.ipynb` — minimal agentic loop reference implementation
- `tool_use/tool_selection_basics.ipynb` — tool schema + handler design fundamentals